# 06 · Three-seed runs for 2D vs 3D occlusion (RQ2)

Repeats notebook 05's experiment with seeds 1 and 2 (seed 0 is already in `results/ep4_b4_s0`), with the
**same code path** (`pose_lib.hmr2_ft` is notebook 05's `finetune_lib`, unchanged by default).

Answers: *Does 3D-rendered occlusion beat 2D-pasted occlusion for fine-tuning?* A difference counts as
**robust** only if it has the same sign in all three seeds **and** the two-level bootstrap CI (resampling
seeds and test samples) excludes zero. Within-seed CIs alone (notebook 05) ignore training noise.

Optional: adds the **per-joint self-occlusion label** to the existing test set (no re-rendering, ~5 min),
so errors split into *visible / hidden by an object / hidden by the body*.

Resumable: every checkpoint and result is on Drive and skipped if present. Runtime on a T4: ~35 min per
seed for training + ~5 min evaluation.

In [ ]:
SEEDS      = [0, 1, 2]
CONDITIONS = ["B_clean", "C_paste2d", "D_occ3d"]
EPOCHS, BATCH, LR, N_BLOCKS = 4, 16, 1e-5, 4      # identical to notebook 05
ADD_SELFOCC = True                                 # add the self-occlusion label to the test set (needs Blender)

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# HMR2.0 without PHALP/detectron2 (the install that worked in notebook 05 on Python 3.13)
%cd /content
![ -d 4D-Humans ] || git clone -q https://github.com/shubham-goel/4D-Humans.git
%cd /content/4D-Humans
!pip install -q numpy yacs smplx pytorch_lightning timm einops omegaconf trimesh webdataset pyrender
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy
!pip install -q --no-deps .
!sed -i 's/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg)/HMR2.load_from_checkpoint(checkpoint_path, strict=False, cfg=model_cfg, weights_only=False)/' hmr2/models/__init__.py
%cd /content
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, "/content/4D-Humans")

from hmr2.configs import CACHE_DIR_4DHUMANS
from hmr2.models import download_models
download_models(CACHE_DIR_4DHUMANS)                           # ~2.5 GB, cached for the session

# SMPL: plain-numpy files only, chumpy never imported (the fix for the broken environment)
from pose_lib.smpl import pkl_to_npz, setup_hmr2_smpl
SMPL_NPZ = f"{PROJECT_DIR}/data/smpl_neutral.npz"
if not os.path.exists(SMPL_NPZ):
    print(pkl_to_npz(f"{PROJECT_DIR}/body_models/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl", SMPL_NPZ))
setup_hmr2_smpl(SMPL_NPZ)

import torch
from pose_lib import hmr2_ft as ft, lora, metrics, stats
fk = ft.SMPLJoints(SMPL_NPZ)
print("HMR2.0 + SMPL ready")

In [ ]:
# synthetic AMASS dataset from notebook 04 (B_clean / C_paste2d / D_occ3d, train + test)
DATA = "/content/dataset"
if not os.path.exists(f"{DATA}/test/D_occ3d/labels.npz"):
    !tar -xf "{PROJECT_DIR}/data/synth_amass.tar" -C /content
# restore self-occlusion labels if notebook 06 added them in an earlier session
for c in ["B_clean", "C_paste2d", "D_occ3d"]:
    saved = f"{PROJECT_DIR}/data/test_labels_selfocc_{c}.npz"
    if os.path.exists(saved):
        shutil.copy(saved, f"{DATA}/test/{c}/labels.npz")
HAS_SELFOCC = "selfvis_joint" in np.load(f"{DATA}/test/D_occ3d/labels.npz").files
print(os.listdir(f"{DATA}/train"), "| self-occlusion labels:", HAS_SELFOCC)

In [ ]:
def fresh_model():
    m = ft.load_model()
    gc.collect(); torch.cuda.empty_cache()
    return m

## Optional: per-joint self-occlusion labels for the test set (labels only, no rendering)

In [ ]:
if ADD_SELFOCC and not HAS_SELFOCC:
    !apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libgl1 libegl1 > /dev/null
    ![ -d /content/blender ] || (wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content && mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz)
    BLENDER = "/content/blender/blender"
    N_TEST, N_PROCS, SEED0 = 1000, 3, 100          # must match notebook 04's test generation
    per = (N_TEST + N_PROCS - 1) // N_PROCS
    procs, gens = [], []
    os.makedirs("/content/selfocc", exist_ok=True)
    for p in range(N_PROCS):
        k = min(per, N_TEST - p * per); out = f"/content/selfocc/test_{p}"
        cmd = [BLENDER, "--background", "--python", f"{CODE_DIR}/blender/synth_gen.py", "--", "--model", SMPL_NPZ,
               "--poses", f"{PROJECT_DIR}/data/poses_test.npz", "--out", out, "--n", str(k), "--start", str(p * per),
               "--seed", str(SEED0 + p), "--samples", "1", "--labels-only", "--selfvis-joints"]
        procs.append(subprocess.Popen(cmd, stdout=open(out + ".log", "w"), stderr=subprocess.STDOUT)); gens.append(out)
    while any(pr.poll() is None for pr in procs):
        time.sleep(30); print(time.strftime("%H:%M"), [open(g + ".log").read().count("[synth]") for g in gens])
    from pose_lib.labels import merge_selfvis
    print(merge_selfvis(gens, [f"{DATA}/test/{c}" for c in CONDITIONS]))   # checks the samples match first
    for c in CONDITIONS:
        shutil.copy(f"{DATA}/test/{c}/labels.npz", f"{PROJECT_DIR}/data/test_labels_selfocc_{c}.npz")
    HAS_SELFOCC = True

## Train (skips anything already on Drive)

In [ ]:
for seed in SEEDS:
    RUN = f"ep{EPOCHS}_b{N_BLOCKS}_s{seed}"
    MODELS = f"{PROJECT_DIR}/models/{RUN}"; os.makedirs(MODELS, exist_ok=True)
    for cond in CONDITIONS:
        ckpt = f"{MODELS}/{cond}.pt"
        if os.path.exists(ckpt):
            print(f"[skip] seed {seed} {cond}"); continue
        print(f"\n===== seed {seed}: training on {cond} =====")
        model = fresh_model()
        hist = ft.train(model, f"{DATA}/train/{cond}", f"{DATA}/train/{cond}", fk, ckpt,
                        epochs=EPOCHS, batch_size=BATCH, lr=LR, n_blocks=N_BLOCKS, seed=seed)
        json.dump(hist, open(f"{MODELS}/{cond}_history.json", "w"), indent=1)
        del model

## Evaluate every model on every test set

In [ ]:
test = {c: ft.SynthDataset(f"{DATA}/test/{c}") for c in CONDITIONS}
print({c: len(d) for c, d in test.items()}, "test samples (occ_ok only)")
names = ["pretrained"] + CONDITIONS
for seed in SEEDS:
    RUN = f"ep{EPOCHS}_b{N_BLOCKS}_s{seed}"
    MODELS, RESULTS = f"{PROJECT_DIR}/models/{RUN}", f"{PROJECT_DIR}/results/{RUN}"
    os.makedirs(RESULTS, exist_ok=True)
    for name in names:
        if name == "pretrained" and seed != SEEDS[0]:
            continue                                   # the released model doesn't depend on the seed
        todo = [t for t in CONDITIONS if not os.path.exists(f"{RESULTS}/{name}__{t}.npz")
                or (HAS_SELFOCC and "selfvis" not in np.load(f"{RESULTS}/{name}__{t}.npz").files)]
        if not todo:
            continue
        model = fresh_model()
        if name != "pretrained":
            ft.load_trainable(model, f"{MODELS}/{name}.pt")
        for t in todo:
            print(f"seed {seed} {name:>10} on {t}:", end=" ")
            np.savez(f"{RESULTS}/{name}__{t}.npz", **ft.evaluate(model, test[t], fk))
        del model

## Results: mean ± std over seeds, robust comparisons

In [ ]:
import pandas as pd
R = {}
for seed in SEEDS:
    RES = f"{PROJECT_DIR}/results/ep{EPOCHS}_b{N_BLOCKS}_s{seed}"
    for n in names:
        src = RES if n != "pretrained" else f"{PROJECT_DIR}/results/ep{EPOCHS}_b{N_BLOCKS}_s{SEEDS[0]}"
        for t in CONDITIONS:
            R[(seed, n, t)] = dict(np.load(f"{src}/{n}__{t}.npz"))

rows = []
for t in CONDITIONS:
    for n in names:
        per = [metrics.summarize(R[(s, n, t)]) for s in SEEDS]
        keys = [k for k in per[0] if k != "n" and all(k in p for p in per)]
        row = {"test": t, "model": n}
        row.update(stats.mean_std_table({k: [p[k] for p in per] for k in keys}))
        rows.append(row)
df = pd.DataFrame(rows).set_index(["test", "model"])
OUTDIR = f"{PROJECT_DIR}/results/seeds_ep{EPOCHS}_b{N_BLOCKS}"; os.makedirs(OUTDIR, exist_ok=True)
df.to_csv(f"{OUTDIR}/summary_mean_std.csv"); pd.set_option("display.width", 250); df

In [ ]:
def cmp(a, b, t, metric="pa_mpjpe"):
    r = stats.seed_comparison([R[(s, a, t)][metric] for s in SEEDS], [R[(s, b, t)][metric] for s in SEEDS])
    verdict = "ROBUST" if r["robust"] else ("same sign, CI includes 0" if r["same_sign_all_seeds"] else "sign flips across seeds")
    return (f"{a} - {b} on {t}: {r['diff_mean']:+.1f} mm (seed sd {r['diff_std_across_seeds']:.1f}; "
            f"per seed {r['per_seed']}) [95% CI {r['ci_lo']:+.1f}, {r['ci_hi']:+.1f}] -> {verdict}")

lines = ([cmp("D_occ3d", "C_paste2d", t) for t in CONDITIONS] +
         [cmp("D_occ3d", "B_clean", t) for t in ("C_paste2d", "D_occ3d")] +
         [cmp("C_paste2d", "B_clean", t) for t in ("C_paste2d", "D_occ3d")] +
         [cmp("D_occ3d", "pretrained", t) for t in CONDITIONS])
print("\n".join(lines)); open(f"{OUTDIR}/comparisons_seeds.txt", "w").write("\n".join(lines))

**How to read RQ2.** The key test is *D beats C on C's own test set* (each model has a home advantage).
The cost on clean images (D − B on B_clean) is the trade-off that motivates specialists.

In [ ]:
import matplotlib.pyplot as plt
bins = [(0, .2), (.2, .4), (.4, .6), (.6, 1.01)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, t in zip(axes, ["C_paste2d", "D_occ3d"]):
    for n, style in zip(names, ["k--", "C0-o", "C1-s", "C2-^"]):
        ys = np.array([[R[(s, n, t)]["pa_mpjpe"][(R[(s, n, t)]["occfrac"] >= lo) & (R[(s, n, t)]["occfrac"] < hi)].mean()
                        for lo, hi in bins] for s in SEEDS])
        ax.errorbar(range(len(bins)), ys.mean(0), yerr=ys.std(0, ddof=1) if len(SEEDS) > 1 else None,
                    fmt=style, capsize=3, label=n)
    ax.set_xticks(range(len(bins))); ax.set_xticklabels([f"{lo:.0%}-{min(hi,1):.0%}" for lo, hi in bins])
    ax.set_title(f"test set: {t}"); ax.set_xlabel("fraction of body hidden"); ax.grid(alpha=.3)
axes[0].set_ylabel("PA-MPJPE (mm), mean ± sd over seeds"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{OUTDIR}/error_vs_occlusion_seeds.png", dpi=150); plt.show()

In [ ]:
# Self-occlusion split (if labels were added): visible vs hidden-by-object vs hidden-by-body, D_occ3d test
if HAS_SELFOCC:
    keys = ["PA joints: visible", "PA joints: hidden by object", "PA joints: hidden by body"]
    tab = {n: stats.mean_std_table({k: [metrics.summarize(R[(s, n, "D_occ3d")]).get(k, np.nan) for s in SEEDS]
                                    for k in keys}) for n in names}
    t = pd.DataFrame(tab).T; t.to_csv(f"{OUTDIR}/selfocc_split_D_occ3d.csv"); display(t)